Техническая реализация и описание ТЗ:
1. Необходимо проанализировать BoM Excel файл, посмотреть что в нем лежит и как с ним работать
2. Предоставить BoM explosion в формате, приведенном выше. Отчет представить в годовом формате (2024, 2025). Произвести explosion по заводу, FIN материалу и году.

In [2]:
import pandas as pd
import numpy as np

In [3]:
bom = pd.read_csv("./data/task_2_data_ex.csv")
bom

,year,month,produced_material,produced_material_production_type,produced_material_release_type,produced_material_quantity,component_material,component_material_production_type,component_material_release_type,component_material_quantity,plant_id
0,2024,1,10000,8002,FIN,990.00,50000,8002.0,PROD,990.00,RLT_10
1,2024,1,50000,8002,PROD,859.00,80070,8007.0,PROD,879.00,RLT_10
2,2024,1,50000,8002,PROD,859.00,90000,NaN,ADD,50.00,RLT_10
3,2024,1,50000,8002,PROD,859.00,90001,NaN,ADD,20.00,RLT_10
4,2024,1,80070,8007,PROD,929.00,80010,8001.0,PROD,"3,626.00",RLT_10
...,...,...,...,...,...,...,...,...,...,...,...
1315,2024,12,80079,8007,PROD,849.00,90048,NaN,ADD,9.00,RLT_14
1316,2024,12,80019,8001,PROD,"1,780.00",80009,8000.0,PROD,"2,046.00",RLT_14
1317,2024,12,80019,8001,PROD,"1,780.00",90049,NaN,ADD,102.00,RLT_14
1318,2024,12,80009,8000,PROD,"2,181.00",70009,NaN,RM,"2,523.00",RLT_14


1. Аналитика сырых данных:

In [4]:
bom.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1320 entries, 0 to 1319
Data columns (total 11 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   year                                1320 non-null   int64  
 1   month                               1320 non-null   int64  
 2   produced_material                   1320 non-null   int64  
 3   produced_material_production_type   1320 non-null   int64  
 4   produced_material_release_type      1320 non-null   object 
 5   produced_material_quantity          1320 non-null   object 
 6   component_material                  1320 non-null   int64  
 7   component_material_production_type  480 non-null    float64
 8   component_material_release_type     1320 non-null   object 
 9   component_material_quantity         1320 non-null   object 
 10  plant_id                            1320 non-null   object 
dtypes: float64(1), int64(5), object(5)
memory u

1. _production_type колонки имеют разные типы данных: int64, float64
2. _quantity колонки имеют тип object, хотя содержат в себе числовые значения
3. Все колонки содержат 1320 non-null записей, только колонка №7 480 non-null записей. Это означает, что 480 строк у нас типа ADD или RM

In [5]:
non_numeric_prod = bom[pd.to_numeric(bom['produced_material_quantity'], errors='coerce').isna()]
print("Строки с некорректным количеством (Produced):")
print(non_numeric_prod['produced_material_quantity'].unique())

Строки с некорректным количеством (Produced):
['1,726.00' '1,980.00' '1,629.00' '1,829.00' '1,954.00' '2,002.00'
 '1,006.00' '1,781.00' '1,955.00' '1,004.00' '1,742.00' '2,077.00'
 '1,730.00' '1,830.00' '1,963.00' '2,157.00' '1,625.00' '2,041.00'
 '1,706.00' '1,846.00' '1,803.00' '1,845.00' '1,705.00' '2,109.00'
 '1,057.00' '1,649.00' '1,807.00' '1,670.00' '1,904.00' '1,050.00'
 '1,635.00' '2,144.00' '1,708.00' '1,854.00' '1,070.00' '1,862.00'
 '2,075.00' '1,630.00' '2,166.00' '1,772.00' '2,185.00' '1,054.00'
 '1,773.00' '1,944.00' '1,044.00' '1,922.00' '1,931.00' '1,707.00'
 '1,950.00' '1,049.00' '1,895.00' '2,045.00' '1,946.00' '1,859.00'
 '1,081.00' '1,780.00' '2,181.00' '1,032.00' '1,825.00' '2,136.00'
 '1,084.00' '1,820.00' '1,005.00' '1,647.00' '1,945.00' '1,755.00'
 '2,023.00' '1,665.00' '2,063.00' '1,979.00' '1,911.00' '1,964.00'
 '1,865.00' '1,034.00' '1,725.00' '1,941.00' '1,040.00' '1,679.00'
 '1,924.00' '1,686.00' '1,972.00' '1,069.00' '1,709.00' '1,887.00'
 '1,956.00' '1,9

Мы видим, что числа больше 1000 неправильно сконвертировались в тип Float из-за наличия запятой, разделяющей разряды числа, соответственно их надо убрать

In [6]:
def exclude_comma(quantity):
    if isinstance(quantity, str):
        return quantity.replace(',', '')
    return quantity

bom['produced_material_quantity'] = bom['produced_material_quantity'].apply(exclude_comma)
bom['produced_material_quantity'] = bom['produced_material_quantity'].astype(float)

bom['component_material_quantity'] = bom['component_material_quantity'].apply(exclude_comma)
bom['component_material_quantity'] = bom['component_material_quantity'].astype(float)


In [7]:
bom.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1320 entries, 0 to 1319
Data columns (total 11 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   year                                1320 non-null   int64  
 1   month                               1320 non-null   int64  
 2   produced_material                   1320 non-null   int64  
 3   produced_material_production_type   1320 non-null   int64  
 4   produced_material_release_type      1320 non-null   object 
 5   produced_material_quantity          1320 non-null   float64
 6   component_material                  1320 non-null   int64  
 7   component_material_production_type  480 non-null    float64
 8   component_material_release_type     1320 non-null   object 
 9   component_material_quantity         1320 non-null   float64
 10  plant_id                            1320 non-null   object 
dtypes: float64(3), int64(5), object(3)
memory u

In [8]:
print("\nРаспределение типов материалов:")
print(bom['produced_material_release_type'].value_counts())


Распределение типов материалов:
produced_material_release_type
PROD    1200
FIN      120
Name: count, dtype: int64


Таким образом мы понимаем, что в таблице есть 120 FIN продуктов на продажу, для которых мы и строим иерархию

In [9]:
print("\nУникальные этапы производства (Production Types):")
print(sorted(bom['produced_material_production_type'].unique()))


Уникальные этапы производства (Production Types):
[np.int64(8000), np.int64(8001), np.int64(8002), np.int64(8007)]


В этапах производства действительно содержатся только числовые значения этапов: 8000 -> 8001 -> 8007 -> 8002

Проверим, сколько готовых продуктов (FIN) на каждом заводе:

In [10]:
print("\nКоличество финальных продуктов по заводам:")
print(bom[bom['produced_material_release_type'] == 'FIN'].groupby('plant_id')['produced_material'].nunique())


Количество финальных продуктов по заводам:
plant_id
RLT_10    2
RLT_14    6
RLT_16    2
Name: produced_material, dtype: int64


2. BoM explosion

- Группируем таблицу по году, заводу, выпущенному материалу и его компоненту
- Суммируем количественные показатели выпуска для того, чтобы узнать общий объем полученного материала и компонентов для его выпуска за год по каждому заводу

In [11]:
bom_annual = bom.groupby([
    'year', 'plant_id', 
    'produced_material', 'produced_material_release_type', 'produced_material_production_type',
    'component_material', 'component_material_release_type', 'component_material_production_type'
], dropna=False).agg({
    'produced_material_quantity': 'sum',
    'component_material_quantity': 'sum'
}).reset_index()

print(bom_annual.head())

   year plant_id  produced_material produced_material_release_type  \
0  2024   RLT_10              10000                            FIN   
1  2024   RLT_10              10001                            FIN   
2  2024   RLT_10              50000                           PROD   
3  2024   RLT_10              50000                           PROD   
4  2024   RLT_10              50000                           PROD   

   produced_material_production_type  component_material  \
0                               8002               50000   
1                               8002               50001   
2                               8002               80070   
3                               8002               90000   
4                               8002               90001   

  component_material_release_type  component_material_production_type  \
0                            PROD                              8002.0   
1                            PROD                              8002.0   

In [12]:
print(len(bom_annual))

110


110 - количество различных комбинаций группировки как по году и заводу, так и связей материал - компонент
Это говорит о том, что финальная таблица тоже должна содержать 110 записей


Рекурсивная функция:

In [13]:
def get_bom_explosion_recursive(current_df, reference_df):
    # Если есть строки, где текущий компонент имеет тип PROD, то мерджим их
    to_explode = current_df[current_df['component_material_release_type'] == 'PROD']
    
    if to_explode.empty:
        return pd.DataFrame()
    
    # Подготавливаем справочник (столбцы (компонента), которые добавляются к записи)
    ref = reference_df.rename(columns={
        'plant_id': 'plant',
        'produced_material': 'prod_material_id',
        'produced_material_release_type': 'prod_material_release_type',
        'produced_material_production_type': 'prod_material_production_type',
        'produced_material_quantity': 'prod_material_production_quantity',
        'component_material': 'component_id',
        'component_material_release_type': 'component_material_release_type',
        'component_material_production_type': 'component_material_production_type',
        'component_material_quantity': 'component_consumption_quantity'
    })
    
    # Соединяем по условию component_id "левой таблицы" должен равняться prod_material_id "правой" таблицы 
    # Столбцы FIN материала, которые есть в каждой записи, к которым добавляется новый материал/компонент (из ref)
    next_gen = to_explode[[
        'plant', 'year', 'fin_material_id', 'fin_material_release_type', 
        'fin_material_production_type', 'fin_production_quantity', 'component_id'
    ]].merge(
        ref,
        left_on=['plant', 'year', 'component_id'],
        right_on=['plant', 'year', 'prod_material_id'],
        how='inner'
    )
    
    if next_gen.empty:
        return pd.DataFrame()
    
    # Убираем дубликат ключа после мерджа
    next_gen = next_gen.drop(columns=['component_id_x']).rename(columns={'component_id_y': 'component_id'})
    
    return pd.concat([next_gen, get_bom_explosion_recursive(next_gen, reference_df)], ignore_index=True)

1. start - таблица со всеми конечными продуктами, где produced_material_release_type = FIN
2. initial_layer - нулевой уровень вложенности для рекурсии и структура финальной таблицы
3. final_report - финальная таблица bom explosion

In [14]:
start_data = bom_annual[bom_annual['produced_material_release_type'] == 'FIN'].copy()

initial_layer = pd.DataFrame({
    'plant': start_data['plant_id'],
    'year': start_data['year'],
    'fin_material_id': start_data['produced_material'],
    'fin_material_release_type': start_data['produced_material_release_type'],
    'fin_material_production_type': start_data['produced_material_production_type'],
    'fin_production_quantity': start_data['produced_material_quantity'],
    # На нулевом этапе родитель - это FIN
    'prod_material_id': start_data['produced_material'],
    'prod_material_release_type': start_data['produced_material_release_type'],
    'prod_material_production_type': start_data['produced_material_production_type'],
    'prod_material_production_quantity': start_data['produced_material_quantity'],
    # А компонент - это первый PROD
    'component_id': start_data['component_material'],
    'component_material_release_type': start_data['component_material_release_type'],
    'component_material_production_type': start_data['component_material_production_type'],
    'component_consumption_quantity': start_data['component_material_quantity']
})

# 3. Запускаем рекурсию и склеиваем результаты
final_report = pd.concat([initial_layer, get_bom_explosion_recursive(initial_layer, bom_annual)], ignore_index=True)

print(f"Строк: {len(final_report)}")

# Проверяем результат для fin_material_id = 10000
final_report[final_report['fin_material_id'] == 10000]

Строк: 110


,plant,year,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_material_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity
0,RLT_10,2024,10000,FIN,8002,11708.0,10000,FIN,8002,11708.0,50000,PROD,8002.0,11708.0
10,RLT_10,2024,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,80070,PROD,8007.0,11303.0
11,RLT_10,2024,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90000,ADD,NaN,598.0
12,RLT_10,2024,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90001,ADD,NaN,242.0
40,RLT_10,2024,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,80010,PROD,8001.0,41769.0
41,RLT_10,2024,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,90002,ADD,NaN,355.0
42,RLT_10,2024,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,90003,ADD,NaN,121.0
70,RLT_10,2024,10000,FIN,8002,11708.0,80010,PROD,8001,21013.0,80000,PROD,8000.0,23360.0
71,RLT_10,2024,10000,FIN,8002,11708.0,80010,PROD,8001,21013.0,90004,ADD,NaN,1242.0
90,RLT_10,2024,10000,FIN,8002,11708.0,80000,PROD,8000,23478.0,70000,RM,NaN,30498.0


Данная выборка fin_material_id 10000 показывает все материалы/компоненты при изготовке данного продукта

Производственная цепочка для данного fin_material_id:

1. 10000 FIN -> 50000 PROD

2. 50000 PROD ->

- 80070 PROD (продолжение цепи)

- 90000 ADD

- 90001 ADD

3. 80070 PROD ->

- 80010 PROD (продолжение цепи)

- 90002 ADD

- 90003 ADD

4. 80010 PROD ->

- 80000 PROD (продолжение цепи)

- 90004 ADD

5. 80000 PROD ->

- 70000 RM

- 90005 ADD


3. SQL-запрос

1. Инициализируем переменные и url для подключения к Базе данных bom_explosion

In [16]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

# Загружаем переменные из файла .env
load_dotenv()

# Достаем данные из окружения
USER = os.getenv('DB_USER')
PASSWORD = os.getenv('DB_PASSWORD')
HOST = os.getenv('DB_HOST')
PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

DATABASE_URL = f'postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}'

2. Подключаемся к БД, создаем новую таблицу bom_table, загружаем сгруппированный датасет bom_annual

In [ ]:
try:
    engine = create_engine(DATABASE_URL)
    
    with engine.connect() as conn:
        print(f"Безопасное подключение к базе '{DB_NAME}' установлено!")
    
    bom_annual.to_sql('bom_table', engine, if_exists='replace', index=False)
    print("Таблица 'bom_table' успешно обновлена.")
    
except Exception as e:
    print(f"Ошибка: {e}")

3. Проверяем данные, которые находятся сейчас в таблице

In [17]:
query = "SELECT * FROM bom_table"

df_from_db = pd.read_sql(query, engine)

print("Информация о таблице:")
df_from_db.info()

print("\nПервые 10 строк данных:")
print(df_from_db.head(10))

Информация о таблице:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 10 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   year                                110 non-null    int64  
 1   plant_id                            110 non-null    object 
 2   produced_material                   110 non-null    int64  
 3   produced_material_release_type      110 non-null    object 
 4   produced_material_production_type   110 non-null    int64  
 5   component_material                  110 non-null    int64  
 6   component_material_release_type     110 non-null    object 
 7   component_material_production_type  40 non-null     float64
 8   produced_material_quantity          110 non-null    float64
 9   component_material_quantity         110 non-null    float64
dtypes: float64(3), int64(4), object(3)
memory usage: 8.7+ KB

Первые 10 строк данных:
   yea

4. Рекурсивный CTE запрос

In [ ]:
sql_query = """
WITH RECURSIVE explosion AS (
    -- 1. Выбираем нулевой уровень (FIN продукты)
    SELECT 
        plant_id,
        year,
        produced_material AS fin_material_id,
        produced_material_release_type AS fin_material_release_type,
        produced_material_production_type AS fin_material_production_type,
        produced_material_quantity AS fin_production_quantity,
        
        produced_material AS prod_material_id,
        produced_material_release_type AS prod_material_release_type,
        produced_material_production_type AS prod_material_production_type,
        produced_material_quantity AS prod_material_production_quantity,
        
        component_material AS component_id,
        component_material_release_type,
        component_material_production_type,
        component_material_quantity AS component_consumption_quantity,
        1 as depth
    FROM bom_table
    WHERE produced_material_release_type = 'FIN'

    UNION ALL

    SELECT 
        b.plant_id,
        b.year,
        cte.fin_material_id,
        cte.fin_material_release_type,
        cte.fin_material_production_type,
        cte.fin_production_quantity,
        
        b.produced_material,
        b.produced_material_release_type,
        b.produced_material_production_type,
        b.produced_material_quantity,

        b.component_material,
        b.component_material_release_type,
        b.component_material_production_type,
        b.component_material_quantity,
        cte.depth + 1
    FROM bom_table b
    INNER JOIN explosion cte ON 
        b.produced_material = cte.component_id AND 
        b.plant_id = cte.plant_id AND 
        b.year = cte.year
    WHERE cte.component_material_release_type = 'PROD'
)
SELECT * FROM explosion 
ORDER BY fin_material_id, depth;
"""

df_sql_final = pd.read_sql(sql_query, engine)

print(f"Строк: {len(df_sql_final)}")
df_sql_final.head(15)

SQL успешно вернул 110 строк.


,plant_id,year,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_material_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity,depth
0,RLT_10,2024,10000,FIN,8002,11708.0,10000,FIN,8002,11708.0,50000,PROD,8002.0,11708.0,1
1,RLT_10,2024,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90001,ADD,NaN,242.0,2
2,RLT_10,2024,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90000,ADD,NaN,598.0,2
3,RLT_10,2024,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,80070,PROD,8007.0,11303.0,2
4,RLT_10,2024,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,90003,ADD,NaN,121.0,3
5,RLT_10,2024,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,80010,PROD,8001.0,41769.0,3
6,RLT_10,2024,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,90002,ADD,NaN,355.0,3
7,RLT_10,2024,10000,FIN,8002,11708.0,80010,PROD,8001,21013.0,90004,ADD,NaN,1242.0,4
8,RLT_10,2024,10000,FIN,8002,11708.0,80010,PROD,8001,21013.0,80000,PROD,8000.0,23360.0,4
9,RLT_10,2024,10000,FIN,8002,11708.0,80000,PROD,8000,23478.0,70000,RM,NaN,30498.0,5


In [20]:
# Проверяем результат для fin_material_id = 10000
df_sql_final[df_sql_final['fin_material_id'] == 10000]

,plant_id,year,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_material_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity,depth
0,RLT_10,2024,10000,FIN,8002,11708.0,10000,FIN,8002,11708.0,50000,PROD,8002.0,11708.0,1
1,RLT_10,2024,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90001,ADD,NaN,242.0,2
2,RLT_10,2024,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90000,ADD,NaN,598.0,2
3,RLT_10,2024,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,80070,PROD,8007.0,11303.0,2
4,RLT_10,2024,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,90003,ADD,NaN,121.0,3
5,RLT_10,2024,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,80010,PROD,8001.0,41769.0,3
6,RLT_10,2024,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,90002,ADD,NaN,355.0,3
7,RLT_10,2024,10000,FIN,8002,11708.0,80010,PROD,8001,21013.0,90004,ADD,NaN,1242.0,4
8,RLT_10,2024,10000,FIN,8002,11708.0,80010,PROD,8001,21013.0,80000,PROD,8000.0,23360.0,4
9,RLT_10,2024,10000,FIN,8002,11708.0,80000,PROD,8000,23478.0,70000,RM,NaN,30498.0,5


Мы видим, что результаты рекурсивной функции на python и рекурсивного CTE SQL-запроса cовпадают. Уровень вложенности можно увидеть в колонке depth, каждый FIN продукт включает в себя максимум 5-ый уровень вложенности